# train_final_model.ipynb — Final YOLOv8 Training (Kaggle)

Great Barrier Reef COTS Detection Project

Wired to Person A's real data pipeline (`dataset.py`). Run cells top to
bottom, in order — each one prepares something the next one needs.

Running directly in the competition notebook — no Kaggle API auth or
download needed, data is already mounted at `/kaggle/input/`.

Turn on **Internet** in the Settings panel (right sidebar) so the
`git clone` and `pip install` cells below can reach GitHub/PyPI.
Also set **Accelerator → GPU T4 x2**.\n\n---\n**Updated:** now uses `yolo11s.pt` (was `yolov8n.pt`), trains for up to 100 epochs with early stopping (was a fixed 8), uses both GPUs, fixes an imgsz/optimizer mismatch from the previous run, and adds augmentation tuned for small underwater objects. See `model.py` for details.

## Step 1 — Get the repo code (`dataset.py`, `model.py`)

Clones fresh every run so you always get the latest pushed code.
Checks out your branch, then merges in `main` locally (inside this
session only — nothing gets pushed anywhere) so you have both your
files and anything Person A has pushed to `main`.

In [1]:
import sys

!rm -rf /kaggle/working/repo
!git clone https://github.com/RuzannaMkhitaryan/great-barrier-reef.git /kaggle/working/repo
!git -C /kaggle/working/repo checkout anahit
!git -C /kaggle/working/repo merge origin/main --no-edit

sys.path.append('/kaggle/working/repo/src')

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 92 (delta 37), reused 64 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 1.71 MiB | 9.64 MiB/s, done.
Resolving deltas: 100% (37/37), done.
Branch 'anahit' set up to track remote branch 'anahit' from 'origin'.
Switched to a new branch 'anahit'
Already up to date.


## Step 2 — Copy `splits.csv` into the working data folder

Adjust the source path below if it moves in the repo — check with
`!find /kaggle/working/repo -name splits.csv` if this cell errors.

In [2]:
!mkdir -p /kaggle/working/data
!cp /kaggle/working/repo/data/splits/splits.csv /kaggle/working/data/splits.csv

## Step 3 — Install dependencies

In [3]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 4.6 MB/s eta 0:00:00


## Step 4 — Imports and config

In [4]:
!ls /kaggle/input

competitions


In [5]:
import os
import glob
import shutil

import dataset
from model import load_model, get_train_config

# ---- CONFIG ----
# Competition data can be nested (e.g. /kaggle/input/competitions/<slug>/),
# so search recursively for train.csv instead of assuming a fixed depth.
_candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
_reef_candidates = [c for c in _candidates if "barrier" in c.lower() or "reef" in c.lower()]
if _reef_candidates:
    _candidates = _reef_candidates
if not _candidates:
    raise FileNotFoundError("No train.csv under /kaggle/input/**. Run `!ls -R /kaggle/input` to check the Input panel, then set COMP_DIR manually.")
COMP_DIR = os.path.dirname(_candidates[0])
print(f"Using competition data at: {COMP_DIR}")

TRAIN_CSV = f"{COMP_DIR}/train.csv"
SPLITS_CSV = "/kaggle/working/data/splits.csv"
RAW_IMAGES_ROOT = f"{COMP_DIR}/train_images"

LABELS_TRAIN_DIR = "/kaggle/working/data/labels/train"
LABELS_VAL_DIR = "/kaggle/working/data/labels/val"
IMAGES_TRAIN_DIR = "/kaggle/working/data/images/train"
IMAGES_VAL_DIR = "/kaggle/working/data/images/val"
DATA_YAML_PATH = "/kaggle/working/data/data.yaml"

OUTPUT_DIR = "/kaggle/working/great-barrier-reef-checkpoints"
RUN_NAME = "final_model_v3"  # new name - v2's 8-epoch run and this 60-epoch run aren't comparable/resumable from each other

# Model checkpoint to start from. To try a different YOLO generation, just
# change this string - load_model() / Ultralytics handle the rest the same
# way for YOLOv8/v9/v10/YOLO11/YOLO26. Avoid yolo12*.pt (Ultralytics flags
# it as unstable to train). Bumped from yolov8n.pt (nano, 3M params) up to
# yolo11s.pt (small, ~9M params) for more capacity on this small-object task.
MODEL_WEIGHTS = "yolo11s.pt"

# Kaggle gave us GPU T4 x2 - actually use both instead of defaulting to one.
DEVICE = [0, 1]

# Person A's experiment found ratio=all (None) gave the best mAP50, but that
# was presumably measured with a long, converged training run. At only 8
# epochs we never converged, so ratio=None (67% background frames in train)
# mostly just slowed early learning. Start with ratio=2.0 for a faster real
# convergence check, then try None again once epochs/imgsz are fixed and you
# have time budget for a longer run.
NEGATIVE_RATIO = 2.0

# Path Ultralytics writes rolling checkpoints to when save_period is set,
# and whether we should resume from one. NOTE: /kaggle/working does NOT
# persist between separate interactive Kaggle sessions - if your session
# gets cut, you need to "Save Version" (or manually download weights/) so
# last.pt survives, then re-upload it as an input to resume in a new
# session. Within a single session, this lets you resume after e.g. an
# accidental interrupt/kernel restart.
LAST_CHECKPOINT = f"{OUTPUT_DIR}/{RUN_NAME}/weights/last.pt"
RESUME = os.path.exists(LAST_CHECKPOINT)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Using competition data at: /kaggle/input/competitions/tensorflow-great-barrier-reef


## Step 5 — Dataset preparation functions

In [6]:
def clean_yolo_dataset():
    """
    Remove any previously generated YOLO dataset folders so every run
    starts clean - avoids stale labels/images from a prior ratio experiment
    leaking into the current run.
    """
    dirs_to_clean = [IMAGES_TRAIN_DIR, IMAGES_VAL_DIR, LABELS_TRAIN_DIR, LABELS_VAL_DIR]
    for directory in dirs_to_clean:
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.makedirs(directory, exist_ok=True)


def build_yolo_dataset(ratio=NEGATIVE_RATIO):
    """
    Runs Person A's full pipeline: load -> filter negatives (train only)
    -> write YOLO labels -> symlink images -> write data.yaml.
    Prepares files on disk for YOLO to read; returns nothing.
    """
    clean_yolo_dataset()

    full_df = dataset.load_data(TRAIN_CSV, SPLITS_CSV)
    train_df = full_df[full_df["split"] == "train"]
    val_df = full_df[full_df["split"] == "val"]  # never filtered - keep val untouched

    train_df_filtered = dataset.filter_negatives(train_df, ratio=ratio)

    print(f"Train frames after filtering: {len(train_df_filtered)} (ratio={ratio})")
    print(f"Val frames (untouched):       {len(val_df)}")

    dataset.write_yolo_labels(train_df_filtered, labels_dir=LABELS_TRAIN_DIR)
    dataset.write_yolo_labels(val_df, labels_dir=LABELS_VAL_DIR)

    dataset.link_images(train_df_filtered, RAW_IMAGES_ROOT, images_dir=IMAGES_TRAIN_DIR)
    dataset.link_images(val_df, RAW_IMAGES_ROOT, images_dir=IMAGES_VAL_DIR)

    dataset.write_data_yaml(DATA_YAML_PATH, IMAGES_TRAIN_DIR, IMAGES_VAL_DIR)

## Step 6 — Training function

In [7]:
def train():
    build_yolo_dataset(ratio=NEGATIVE_RATIO)

    if RESUME:
        # Ultralytics restores imgsz/epochs/optimizer/augmentation/etc from
        # that run's saved args.yaml automatically - don't pass them again.
        print(f"Found existing checkpoint at {LAST_CHECKPOINT} - resuming from there.")
        model = load_model(pretrained_weights=LAST_CHECKPOINT)
        results = model.train(resume=True)
    else:
        model = load_model(pretrained_weights=MODEL_WEIGHTS)

        # epochs=60 (was 8): the 8-epoch run\'s loss was still dropping hard
        # and mAP was still noisy/non-monotonic (close_mosaic=10 never even
        # triggered) - 8 epochs was cut way too short.
        # batch_size=24 (was 16): the 8-epoch run only used ~8.6GB of the
        # 14.9GB on each T4, so there\'s room to raise it. Drop back to 16
        # if you hit an out-of-memory error.
        # save_period=5: writes epoch5.pt, epoch10.pt, ... to weights/ in
        # addition to last.pt/best.pt, so a long run can survive a Kaggle
        # session interruption - see RESUME logic above.
        config = get_train_config(
            image_size=1280,
            epochs=60,
            batch_size=24,
            learning_rate=0.01,
            patience=10,
            optimizer="SGD",
            device=DEVICE,
            save_period=5,
        )
        print("Requested train config:", config)


        results = model.train(
            data=DATA_YAML_PATH,
            imgsz=config["imgsz"],
            epochs=config["epochs"],
            batch=config["batch"],
            optimizer=config["optimizer"],
            lr0=config["lr0"],
            patience=config["patience"],
            device=config["device"],
            save_period=config["save_period"],
            mosaic=config["mosaic"],
            mixup=config["mixup"],
            copy_paste=config["copy_paste"],
            hsv_h=config["hsv_h"],
            hsv_s=config["hsv_s"],
            hsv_v=config["hsv_v"],
            project=OUTPUT_DIR,
            name=RUN_NAME,
            workers=4,
            cache="disk",
            plots=True,
        )

    # Sanity check: confirm what Ultralytics actually trained with matches
    # what we asked for.
    actual = model.trainer.args
    print(f"Actual imgsz used:     {actual.imgsz}")
    print(f"Actual optimizer used: {actual.optimizer}")
    print(f"Actual lr0 used:       {actual.lr0}")
    print(f"Actual epochs used:    {actual.epochs}")
    print(f"Actual save_period:    {actual.save_period}")

    print("Training complete.")
    print(f"Results and checkpoints saved to: {OUTPUT_DIR}/{RUN_NAME}")
    return results

## Step 7 — Run training

Checkpoints save to `/kaggle/working/`, so they persist as notebook
output when you click **Save Version**.

In [8]:
results = train()

Train frames after filtering: 12846 (ratio=2.0)
Val frames (untouched):       4057
Requested train config: {'imgsz': 1280, 'epochs': 60, 'batch': 24, 'lr0': 0.01, 'patience': 10, 'optimizer': 'SGD', 'device': [0, 1], 'save_period': 5, 'mosaic': 1.0, 'mixup': 0.1, 'copy_paste': 0.1, 'hsv_h': 0.015, 'hsv_s': 0.5, 'hsv_v': 0.3}
Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dyn

In [2]:
!ls /kaggle/working/great-barrier-reef-checkpoints/final_model_v3/weights/best.pt

ls: cannot access '/kaggle/working/great-barrier-reef-checkpoints/final_model_v3/weights/best.pt': No such file or directory


In [1]:
from metrics import f2_from_results_csv

f2_from_results_csv("outputs/yolo11s/results.csv")

ModuleNotFoundError: No module named 'metrics'